Week 15 · Day 1 — Why RAG? The Pipeline
Why this matters

LLMs are powerful but forget facts and hallucinate. Retrieval-Augmented Generation (RAG) fixes this by letting models look things up in an external knowledge base before answering — critical for real-world apps like chatbots, legal research, and enterprise tools.

Theory Essentials

Problem: LLMs have limited context + outdated training.

RAG: Combine retrieval (search) + generation (LLM).

Pipeline:

Embed documents into vectors.

Store embeddings in a database.

Retrieve top-k relevant docs for a query.

Generate an answer with the LLM, grounded in docs.

Benefit: Reduces hallucination, enables domain-specific Q&A.

Limitation: Retrieval quality depends on embeddings and indexing.

How it works (the pipeline)

Think of it as 4 steps:

Embed the documents

Each text chunk is turned into a vector (list of numbers) that captures its meaning.

Example: "Paris is the capital of France" → [0.12, -0.33, 0.88, ...]

We usually use SentenceTransformers or OpenAI embeddings.

Store in a vector database

All embeddings are stored in a structure that allows fast similarity search.

Common tools: FAISS, Chroma, Pinecone, Weaviate.

Retrieve at query time

User asks: “What is the capital of France?”

System converts this question into an embedding, then finds the most similar document vectors.

It retrieves, say, the chunk about “Paris is the capital of France”.

Generate with context

Instead of sending the raw question to the LLM, we send:

Context: Paris is the capital of France.
Question: What is the capital of France?


The LLM answers grounded in the context → “Paris”.

In [2]:
# Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Example: tiny knowledge base
docs = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Madrid is the capital of Spain.",
    "Rome is the capital of Italy."
]

# 1. Embed
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(docs)

# 2. Query
query = "What is the capital of Germany?"
q_emb = model.encode([query])

# 3. Retrieve (cosine similarity)
sims = cosine_similarity(q_emb, embeddings)[0]
top_idx = np.argmax(sims)

print("Query:", query)
print("Top Retrieved Doc:", docs[top_idx])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\AI-Mastery\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: What is the capital of Germany?
Top Retrieved Doc: Berlin is the capital of Germany.


1) Core (10–15 min)

Task: Change the query to ask about Spain.

In [3]:
query = "What is the capital of Spain?"
q_emb = model.encode([query])
sims = cosine_similarity(q_emb, embeddings)[0]
print(docs[np.argmax(sims)])


Madrid is the capital of Spain.


2) Practice (10–15 min)

Task: Return the top-2 most similar documents instead of just one.

In [4]:
top2 = np.argsort(sims)[-2:][::-1]
for i in top2:
    print(docs[i], "→ score:", sims[i])


Madrid is the capital of Spain. → score: 0.8137282
Rome is the capital of Italy. → score: 0.46259356


3) Stretch (optional, 10–15 min)

Task: Add a new doc about Lisbon and test queries.

In [5]:
docs.append("Lisbon is the capital of Portugal.")
embeddings = model.encode(docs)  # re-embed
query = "Capital of Portugal?"
q_emb = model.encode([query])
sims = cosine_similarity(q_emb, embeddings)[0]
print(docs[np.argmax(sims)])


Lisbon is the capital of Portugal.


Mini-Challenge (≤40 min)

Build a Tiny RAG Prototype

Add at least 6 new docs (your choice: countries, movies, tech).

Write a function answer(query, docs) that:

Retrieves top-3 docs.

Returns them as “context” for an LLM (here: just print).

Acceptance Criteria:

Function works for multiple queries.

Retrieval consistently surfaces the correct doc(s).

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

docs = [
    "Ilia Topuria is the pound-for-pound number one.",
    "Ilia Topuria is the UFC lightweight champion.",
    "Alexander Volkanovski is the UFC featherweight champion.",
    "Merab Dvalishvili is the UFC bantamweight champion.",
    "Khamzat Chimaev is the UFC middleweight champion.",
    "Tom Aspinall is the UFC heavyweight champion.",
    "Conor McGregor is a former two-division UFC champion at featherweight and lightweight."
]

# Embed docs once
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embs = model.encode(docs, convert_to_tensor=True)

def answer(query, k=3):
    # Encode query
    q_emb = model.encode([query], convert_to_tensor=True)
    # Similarities
    sims = util.cos_sim(q_emb, doc_embs)[0].cpu().numpy()
    # Top-k indices (sorted by similarity)
    topk = np.argsort(sims)[-k:][::-1]
    
    # Print context (what you'd feed to an LLM)
    print("Question:", query)
    print("Context:")
    for i in topk:
        print("-", docs[i])

# Example
answer("Who is the best?")
answer("Who is Merab?")


Question: Who is the best?
Context:
- Ilia Topuria is the UFC lightweight champion.
- Ilia Topuria is the pound-for-pound number one.
- Khamzat Chimaev is the UFC middleweight champion.
Question: Who is Merab?
Context:
- Merab Dvalishvili is the UFC bantamweight champion.
- Ilia Topuria is the pound-for-pound number one.
- Ilia Topuria is the UFC lightweight champion.


Notes / Key Takeaways

RAG = Embed → Store → Retrieve → Generate.

Retrieval augments memory → avoids hallucinations.

Quality depends on embeddings + similarity search.

Today: simple in-memory index; later: vector DBs (FAISS, Chroma).

This is the foundation of chat-with-your-documents apps.

Reflection

When might a pure LLM be enough without RAG?

What types of data in your life/work would benefit from RAG?


**When might a pure LLM be enough without RAG?**

* When the question is **general knowledge** already covered in the model’s training (e.g., “What is the capital of France?”).
* When the task is **creative generation** (storytelling, brainstorming, coding patterns) that doesn’t need exact facts.
* For **reasoning tasks** where external sources don’t add much, like solving math puzzles or explaining concepts.

**What types of data in your life/work would benefit from RAG?**

* **Company/legal documents** (contracts, agreements, reports) → quick Q&A on specifics.
* **University notes or textbooks** → ask questions over class material.
* **Project docs & emails** → find relevant decisions or details.
* **Technical references** (API docs, research papers) → get grounded answers instead of relying on memory.

---